# 当前美国期权市场 Underlyings

识别 Massive/OPRA 美国上市期权中具有实际交易意义的 underlying：先从最近完整交易日筛出有成交的合约，再用最新快照确认至少一个代表合约仍有有效双边报价。休市时结果表示“最近报价确认、下一交易时段候选”，不会误称此刻能够立即成交。

In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
from hashlib import sha256
import csv
import gzip
import json
import os
import sys
import re

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'kairospy').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

AS_OF = date.today().isoformat()
CACHE = ROOT / 'data' / 'reference' / 'provider=massive' / 'tradable-option-underlyings' / f'as_of={AS_OF}' / 'contracts.json'
print({'as_of': AS_OF, 'cache': str(CACHE)})

## 1. 产生候选并验证最新报价

候选集来自本地最近一份 OPRA Day Aggregates：它证明合约在最近完整交易日有成交。每个 underlying 选择成交量最高的 3 个未到期合约，通过 `/v3/snapshot` 每批 200 个拉取最新快照；至少一个代表合约具有 `0 ≤ bid ≤ ask` 且 `ask > 0` 才保留。批次独立缓存，可断点续跑。该抽样可能漏掉“前三名代表合约都暂时无报价、其他合约有报价”的极端情况，可提高 `CONTRACTS_PER_UNDERLYING` 降低假阴性。

In [ ]:
from kairospy.integrations.connectors.massive import MassiveClient, MassiveConfig

REFRESH = False
CONTRACTS_PER_UNDERLYING = 3
BATCH_SIZE = 200
if CACHE.exists() and not REFRESH:
    payload = json.loads(CACHE.read_text(encoding='utf-8'))
    contracts = payload['contracts']
    retrieved_at = payload['retrieved_at']
    activity_date = payload['activity_date']
    market_status = payload['market_status']
    source = 'cache'
else:
    if not os.environ.get('MASSIVE_API_KEY'):
        raise RuntimeError('请先在 shell 中设置 MASSIVE_API_KEY，再启动 Jupyter。')
    base_config = MassiveConfig.from_env()
    client = MassiveClient(MassiveConfig(
        api_key=base_config.api_key, timeout_seconds=120, max_retries=base_config.max_retries,
    ))

    files = list((ROOT / 'data/source/provider=massive/resource=flat-files').glob('request_id=*/????-??-??.csv.gz'))
    files = [path for path in files if path.stem.removesuffix('.csv') <= AS_OF
             and 'us_options_opra/' in json.loads((path.parent / 'receipt.json').read_text()).get('file_key', '')]
    if not files:
        raise FileNotFoundError('缺少 OPRA Day Aggregates；请先用 provider-flat-file --provider massive 下载最近完整交易日。')
    activity_file = max(files, key=lambda path: path.stem.removesuffix('.csv'))
    activity_date = activity_file.stem.removesuffix('.csv')

    occ = re.compile(r'^O:([A-Z0-9.]+)(\d{6})([CP])(\d{8})$')
    activity = []
    with gzip.open(activity_file, 'rt', encoding='utf-8', newline='') as handle:
        for row in csv.DictReader(handle):
            match = occ.match(row['ticker'])
            if not match:
                continue
            expiry = datetime.strptime(match.group(2), '%y%m%d').date()
            if expiry < date.fromisoformat(AS_OF):
                continue
            activity.append({
                'ticker': row['ticker'], 'underlying_ticker': match.group(1),
                'expiration_date': expiry.isoformat(),
                'contract_type': 'call' if match.group(3) == 'C' else 'put',
                'volume': float(row.get('volume') or 0),
                'transactions': int(float(row.get('transactions') or 0)),
            })
    candidates = (pd.DataFrame(activity).sort_values(['underlying_ticker', 'volume'], ascending=[True, False])
                  .groupby('underlying_ticker', group_keys=False).head(CONTRACTS_PER_UNDERLYING))
    candidate_by_ticker = candidates.set_index('ticker').to_dict('index')
    tickers = candidates['ticker'].tolist()

    shard_root = CACHE.parent / 'snapshot-batches'
    shard_root.mkdir(parents=True, exist_ok=True)
    snapshots = []
    batches = [tickers[i:i + BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]
    for index, batch in enumerate(batches, 1):
        digest = sha256(','.join(batch).encode()).hexdigest()[:20]
        shard_path = shard_root / f'batch-{digest}.json'
        if shard_path.exists() and not REFRESH:
            rows = json.loads(shard_path.read_text(encoding='utf-8'))
            shard_source = 'cache'
        else:
            rows = []
            for _, page in client.pages('/v3/snapshot', {
                'ticker.any_of': ','.join(batch), 'type': 'options', 'limit': 250,
            }):
                rows.extend(page.get('results', []))
            shard_path.write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
            shard_source = 'api'
        snapshots.extend(rows)
        print(f'[{index}/{len(batches)}] {len(rows)} snapshots ({shard_source})')

    contracts = []
    for snapshot in snapshots:
        ticker = snapshot.get('ticker') or snapshot.get('details', {}).get('ticker')
        quote = snapshot.get('last_quote') or {}
        bid, ask = quote.get('bid'), quote.get('ask')
        if ticker in candidate_by_ticker and bid is not None and ask is not None and 0 <= bid <= ask and ask > 0:
            contracts.append({**candidate_by_ticker[ticker], 'ticker': ticker,
                              'bid': bid, 'ask': ask, 'quote_timestamp': quote.get('last_updated')})
    market_status = client.get('/v1/marketstatus/now').json()
    retrieved_at = datetime.now(timezone.utc).isoformat()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps({
        'provider': 'massive', 'market': 'OPRA', 'as_of': AS_OF, 'activity_date': activity_date,
        'market_status': market_status, 'method': 'recent-activity-plus-latest-two-sided-quote',
        'retrieved_at': retrieved_at, 'contracts': contracts,
    }, ensure_ascii=False), encoding='utf-8')
    source = 'api'

print({'source': source, 'activity_date': activity_date, 'retrieved_at': retrieved_at,
       'market_status': market_status, 'quote_confirmed_contracts': len(contracts)})

## 2. Underlying 清单与覆盖统计

In [ ]:
required = {'ticker', 'underlying_ticker', 'contract_type', 'expiration_date'}
missing = required - set().union(*(row.keys() for row in contracts)) if contracts else required
if missing:
    raise ValueError(f'合约结果缺少字段: {sorted(missing)}')

frame = pd.DataFrame(contracts)
frame['expiration_date'] = pd.to_datetime(frame['expiration_date'], errors='coerce')
frame['contract_type'] = frame['contract_type'].str.lower()
underlyings = (frame.groupby('underlying_ticker', as_index=False)
    .agg(contracts=('ticker', 'nunique'),
         calls=('contract_type', lambda s: int((s == 'call').sum())),
         puts=('contract_type', lambda s: int((s == 'put').sum())),
         recent_volume=('volume', 'sum'),
         first_expiry=('expiration_date', 'min'),
         last_expiry=('expiration_date', 'max'))
    .sort_values(['contracts', 'underlying_ticker'], ascending=[False, True])
    .reset_index(drop=True))

display(pd.DataFrame([{
    'as_of': AS_OF, 'retrieved_at_utc': retrieved_at,
    'activity_date': activity_date, 'quote_confirmed_underlyings': len(underlyings),
    'quote_confirmed_sample_contracts': frame['ticker'].nunique(),
    'earliest_expiry': frame['expiration_date'].min(),
    'latest_expiry': frame['expiration_date'].max(),
}]))
display(underlyings)

## 3. 搜索和快速检查

In [ ]:
QUERY = ''  # 例如 SPX、NVDA；留空显示合约数最多的 50 个
matches = underlyings[underlyings['underlying_ticker'].str.contains(QUERY, case=False, regex=False)]
display(matches.head(50))

ax = underlyings.head(25).sort_values('contracts').plot.barh(
    x='underlying_ticker', y='contracts', figsize=(10, 8), legend=False, color='#2774AE'
)
ax.set(title=f'Active option contracts by underlying ({AS_OF})', xlabel='Contracts', ylabel='Underlying')
plt.tight_layout()

## 4. 导出清单

CSV 与 notebook 使用同一查询日期，便于后续研究直接复用。

In [ ]:
OUTPUT = CACHE.with_name('underlyings.csv')
underlyings.to_csv(OUTPUT, index=False)
print(OUTPUT)